# Comparing defective crystals with symmetry using [`spglib`](https://spglib.readthedocs.io/en/stable/)

In this notebook we study a subtle but very important point in atomistic modelling:

two defective crystals may **look different as generated structures**, but still be **equivalent by symmetry**.  
If we do not detect this, we may think that we produced many different statistical samples, while in fact several of them represent the same configuration.

This matters especially when defective structures are generated automatically and then optimized with atomistic methods.  
Before spending computational effort, it is useful to remove symmetry-equivalent duplicates.

In [ ]:
%load_ext aiida
%aiida

## Learning goals

By the end of this notebook, students should be able to:

- extract symmetry operations from a crystal using `spglib`,
- understand why defective structures must be compared using the **symmetry of the pristine parent crystal**,
- recognize that small random displacements do not necessarily create a new independent defect configuration,
- use symmetry to identify which randomly generated structures are truly distinct.

In [ ]:
import itertools
from collections import defaultdict

import numpy as np
from scipy.optimize import linear_sum_assignment

from ase.spacegroup import crystal
from ase.io import read

import spglib
import aiidalab_widgets_base as awb
from IPython.display import display

In [ ]:
structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        awb.BasicCellEditor(),
    ],
    storable=True,
    node_class="StructureData",
)

def show_structure(structure):
    """Display an ASE structure in the AiiDAlab viewer."""
    structure_selector.structure = structure
    return structure

## Why this problem is non-trivial

Suppose we build a defective crystal, for example by replacing one or more atoms.  
If we apply a symmetry operation of the **pristine crystal**, one defect arrangement may transform into another one.

So the practical question is not:

> "Are the Cartesian coordinates exactly equal?"

but rather:

> "Can one structure be mapped onto the other by a symmetry operation of the pristine crystal, within a reasonable tolerance?"

This is especially important when:
- defects are introduced at random positions among symmetry-equivalent sites,
- atoms are slightly displaced by noise or by a pre-relaxation step,
- we want a statistically meaningful set of independent defective configurations.

<img src="crystal_equivalence.png" width="850">

## Symmetry information from `spglib`

We first define a few helper functions.

The key idea is:

1. obtain the symmetry operations of the pristine crystal from `spglib`,
2. apply each operation to one defective structure,
3. check whether the transformed structure matches the other one.

The matching must be done carefully:
- atoms of the same chemical species may be permuted,
- coordinates must be compared under periodic boundary conditions,
- small displacements should be tolerated.

In [ ]:
def get_symmetry_dataset(ase_atoms, symprec=1e-5, angle_tolerance=-1.0):
    """Return the symmetry dataset of an ASE Atoms object."""
    return spglib.get_symmetry_dataset(
        (
            ase_atoms.get_cell(),
            ase_atoms.get_scaled_positions(),
            ase_atoms.get_atomic_numbers(),
        ),
        symprec=symprec,
        angle_tolerance=angle_tolerance,
    )


def wrap_fractional(frac):
    """Wrap fractional coordinates to the [0, 1) interval."""
    return np.mod(frac, 1.0)


def apply_symmetry_op(frac_positions, rotation, translation):
    """
    Apply a space-group operation to fractional coordinates.

    spglib uses x' = R x + t in fractional coordinates.
    """
    transformed = frac_positions @ rotation.T + translation
    return wrap_fractional(transformed)


def minimum_image_cartesian_differences(frac_a, frac_b, cell, pbc=True):
    """
    Return the Cartesian difference vectors between two sets of fractional coordinates
    using the minimum-image convention.
    """
    diff = frac_a[:, None, :] - frac_b[None, :, :]

    if np.isscalar(pbc):
        pbc = np.array([pbc, pbc, pbc], dtype=bool)
    else:
        pbc = np.array(pbc, dtype=bool)

    for axis in range(3):
        if pbc[axis]:
            diff[..., axis] -= np.round(diff[..., axis])

    return diff @ np.array(cell)


def distance_matrix_pbc(frac_a, frac_b, cell, pbc=True):
    """Pairwise distance matrix under periodic boundary conditions."""
    dcart = minimum_image_cartesian_differences(frac_a, frac_b, cell, pbc=pbc)
    return np.linalg.norm(dcart, axis=-1)


def structures_match_under_permutation(atoms_a, atoms_b, tolerance=0.15):
    """
    Check whether two structures are equivalent up to a permutation of atoms
    of the same chemical species.

    This function assumes that the two structures already have the same cell.
    """
    if len(atoms_a) != len(atoms_b):
        return False

    numbers_a = np.array(atoms_a.get_atomic_numbers())
    numbers_b = np.array(atoms_b.get_atomic_numbers())

    if sorted(numbers_a.tolist()) != sorted(numbers_b.tolist()):
        return False

    frac_a = wrap_fractional(atoms_a.get_scaled_positions())
    frac_b = wrap_fractional(atoms_b.get_scaled_positions())
    cell = atoms_a.get_cell()
    pbc = atoms_a.get_pbc()

    for Z in sorted(set(numbers_a)):
        ia = np.where(numbers_a == Z)[0]
        ib = np.where(numbers_b == Z)[0]

        if len(ia) != len(ib):
            return False

        dmat = distance_matrix_pbc(frac_a[ia], frac_b[ib], cell=cell, pbc=pbc)
        row_ind, col_ind = linear_sum_assignment(dmat)

        if np.any(dmat[row_ind, col_ind] > tolerance):
            return False

    return True


def equivalent_by_pristine_symmetry(pristine, atoms_a, atoms_b, symprec=1e-5, match_tolerance=0.15):
    """
    Check whether atoms_a and atoms_b are equivalent under the symmetry
    operations of the pristine parent crystal.

    This is the natural viewpoint for substitutional defect sampling:
    the defects are compared in the symmetry frame of the pristine lattice.
    """
    dataset = get_symmetry_dataset(pristine, symprec=symprec)
    frac_a = wrap_fractional(atoms_a.get_scaled_positions())

    for rotation, translation in zip(dataset.rotations, dataset.translations):
        transformed = atoms_a.copy()
        transformed.set_scaled_positions(apply_symmetry_op(frac_a, rotation, translation))

        if structures_match_under_permutation(
            transformed,
            atoms_b,
            tolerance=match_tolerance,
        ):
            return True

    return False

## Example 1: one substitutional defect in a simple crystal

We start with a fluorite-like ZrO$_2$ crystal and substitute one Zr atom with Hf.

The two structures below are defective in different places, but the parent crystal has non-trivial symmetry.  
The question is: are these two defect arrangements genuinely different, or are they symmetry-equivalent?

In [ ]:
pristine_zro2 = crystal(
    "ZrO",
    [(0, 0, 0), (3.0 / 4.0, 1.0 / 4.0, 3.0 / 4.0)],
    spacegroup=225,
    cellpar=[[5.09, 0, 0], [0, 5.09, 0], [0, 0, 5.09]],
).repeat([2, 1, 1])

z1 = pristine_zro2.copy()
z2 = pristine_zro2.copy()

z1[0].symbol = "Hf"
z2[3].symbol = "Hf"

dataset_zro2 = get_symmetry_dataset(pristine_zro2)
print("Pristine space group:", dataset_zro2.international)
print("Number of symmetry operations:", len(dataset_zro2.rotations))

In [ ]:
display(structure_selector)

In [ ]:
print("z1")
show_structure(z1)

In [ ]:
get_symmetry_dataset(z1, symprec=1e-5, angle_tolerance=-1.0)

In [ ]:
print("z2")
show_structure(z2)

In [ ]:
equivalent_by_pristine_symmetry(pristine_zro2, z1, z2, match_tolerance=1e-3)

The result above tells us whether the two substitutions represent the same configuration class under the symmetry of the pristine crystal.

For perfectly ideal structures, a very small tolerance is enough.  
For structures with small distortions, we need a larger matching tolerance.

## Example 2: the same defect motif, but with small random displacements

In practice, defective structures are often not perfectly ideal:
- we may add a small random displacement,
- a pre-relaxation may slightly distort local coordinates,
- numerical noise may move atoms a little.

Such small changes should **not** automatically create a new statistical sample.

In [ ]:
rng = np.random.default_rng(7)

z3 = z1.copy()
z4 = z2.copy()

# Add a small random displacement to the substituted atom only.
for structure in [z3, z4]:
    hf_indices = [atom.index for atom in structure if atom.symbol == "Hf"]
    for idx in hf_indices:
        structure.positions[idx] += rng.normal(scale=0.03, size=3)

print("Very strict tolerance:", equivalent_by_pristine_symmetry(pristine_zro2, z3, z4, match_tolerance=1e-3))
print("More realistic tolerance:", equivalent_by_pristine_symmetry(pristine_zro2, z3, z4, match_tolerance=0.15))

This is a useful lesson:

- if the tolerance is too small, two symmetry-equivalent structures may look different just because of tiny displacements;
- if the tolerance is too large, genuinely different structures may be merged incorrectly.

So the tolerance should be chosen with chemical and structural common sense.

## A second point of view: compare **defect sites**, not only full coordinates

For substitutional defects, there is a very natural strategy:

1. start from the pristine crystal,
2. identify the crystallographic sites that may host the substitution,
3. map each defect in the defective structure back to the closest pristine site,
4. compare the resulting set of defect sites under the pristine symmetry operations.

This is often more robust than comparing all atomic coordinates directly, especially when the defect atoms have been slightly displaced.

In [ ]:
def substitution_site_indices(pristine, defective, pristine_symbol, defect_symbol, site_tolerance=0.6):
    """
    Map each substitutional defect atom in 'defective' to the closest pristine site
    of type 'pristine_symbol'.

    Example:
        pristine_symbol='O', defect_symbol='N'
    for O -> N substitution.
    """
    pristine_indices = [atom.index for atom in pristine if atom.symbol == pristine_symbol]
    defect_indices = [atom.index for atom in defective if atom.symbol == defect_symbol]

    pristine_frac = wrap_fractional(pristine.get_scaled_positions()[pristine_indices])
    defect_frac = wrap_fractional(defective.get_scaled_positions()[defect_indices])

    dmat = distance_matrix_pbc(defect_frac, pristine_frac, pristine.get_cell(), pristine.get_pbc())
    row_ind, col_ind = linear_sum_assignment(dmat)

    if np.any(dmat[row_ind, col_ind] > site_tolerance):
        raise ValueError(
            "At least one defect atom could not be matched reliably to a pristine site. "
            "Increase 'site_tolerance' only if this is chemically justified."
        )

    matched_global_indices = [pristine_indices[j] for j in col_ind]
    return sorted(matched_global_indices)


def canonical_defect_signature(pristine, defect_site_indices, site_symbol, symprec=1e-5, site_tolerance=1e-3):
    """
    Build a canonical symmetry-invariant signature for a set of defect sites.

    The idea is to transform the defect sites by all symmetry operations of the pristine crystal
    and keep the lexicographically smallest mapped site tuple.
    """
    dataset = get_symmetry_dataset(pristine, symprec=symprec)

    candidate_sites = [atom.index for atom in pristine if atom.symbol == site_symbol]
    candidate_frac = wrap_fractional(pristine.get_scaled_positions()[candidate_sites])
    defect_local = [candidate_sites.index(i) for i in defect_site_indices]
    defect_frac = candidate_frac[defect_local]

    signatures = []
    for rotation, translation in zip(dataset["rotations"], dataset["translations"]):
        transformed_defect_frac = apply_symmetry_op(defect_frac, rotation, translation)
        dmat = distance_matrix_pbc(
            transformed_defect_frac,
            candidate_frac,
            pristine.get_cell(),
            pristine.get_pbc(),
        )
        row_ind, col_ind = linear_sum_assignment(dmat)

        if np.any(dmat[row_ind, col_ind] > site_tolerance):
            continue

        mapped_global = tuple(sorted(candidate_sites[j] for j in col_ind))
        signatures.append(mapped_global)

    if not signatures:
        raise ValueError("Could not build a canonical defect signature from the provided sites.")

    return min(signatures)

## Example 3: O $
\rightarrow$ N substitutions in quartz

We now move to a more realistic example.  
Starting from a quartz structure, we create several random structures by replacing two oxygen atoms with nitrogen.

This imitates the usual situation in defect sampling:
many random draws are generated, but some of them may be equivalent by symmetry and should therefore be counted only once.

In [ ]:
pristine = read("./quartz_alpha.xyz")
show_structure(pristine)

In [ ]:
oxygen_indices = [atom.index for atom in pristine if atom.symbol == "O"]
print("Number of oxygen sites available for substitution:", len(oxygen_indices))

### Generate a pool of random structures

For pedagogical purposes we also add a small random displacement to the substituted N atoms.  
This mimics the fact that real generated structures are often not perfectly ideal.

In [ ]:
rng = np.random.default_rng(123)

nstructures = 100
nreplace = 2
displacement_scale = 0.04  # Å

all_structures = []
site_sets = []

for _ in range(nstructures):
    chosen_sites = sorted(rng.choice(oxygen_indices, size=nreplace, replace=False).tolist())

    new_geo = pristine.copy()
    symbols = new_geo.get_chemical_symbols()

    for idx in chosen_sites:
        symbols[idx] = "N"
    new_geo.set_chemical_symbols(symbols)

    # Small random displacement of the substituted atoms.
    for idx in chosen_sites:
        new_geo.positions[idx] += rng.normal(scale=displacement_scale, size=3)

    all_structures.append(new_geo)
    site_sets.append(chosen_sites)

print(f"Generated {len(all_structures)} random defective structures.")

### Check equivalence using the full structure comparison

This approach is general and does not assume that we already know which sites were replaced.  
It is therefore conceptually broad, but it is also a bit heavier computationally.

In [ ]:
equivalent_pairs = []

for i in range(len(all_structures)):
    for j in range(i + 1, len(all_structures)):
        if equivalent_by_pristine_symmetry(
            pristine,
            all_structures[i],
            all_structures[j],
            match_tolerance=0.20,
        ):
            equivalent_pairs.append((i, j))

print("Number of equivalent pairs found:", len(equivalent_pairs))
equivalent_pairs[:5]

### Check equivalence using canonical defect signatures

For substitutional defects, this is often the most transparent approach.  
Each defective structure is mapped back to a set of pristine sites, and this set is reduced to a canonical representative under the symmetry of the pristine crystal.

In [ ]:
defect_signatures = []
for structure in all_structures:
    defect_sites = substitution_site_indices(
        pristine,
        structure,
        pristine_symbol="O",
        defect_symbol="N",
        site_tolerance=0.7,
    )
    signature = canonical_defect_signature(
        pristine,
        defect_sites,
        site_symbol="O",
        site_tolerance=1e-2,
    )
    defect_signatures.append(signature)

signature_groups = defaultdict(list)
for i, signature in enumerate(defect_signatures):
    signature_groups[signature].append(i)

unique_groups = list(signature_groups.values())

print("Total random structures:", len(all_structures))
print("Distinct symmetry-inequivalent configurations:", len(unique_groups))
print("Sizes of the first few equivalence classes:", [len(g) for g in unique_groups[:10]])

The important conceptual point is that **random generation does not automatically imply independent statistical sampling**.  
Several generated structures may correspond to the same symmetry class.

In practice, this means:

- generate many candidates,
- identify duplicates by symmetry,
- keep only one representative of each symmetry-inequivalent class,
- then optimize those representatives and compare their energies.

In [ ]:
# Display one equivalence class, if any non-trivial class was found.
nontrivial_groups = [group for group in unique_groups if len(group) > 1]
nontrivial_groups[:5]

In [ ]:
if nontrivial_groups:
    group = nontrivial_groups[0]
    print("Showing two symmetry-equivalent members of the same class:", group[:2])

    i, j = group[0], group[1]
    print(f"Structure {i}, substituted sites drawn originally: {site_sets[i]}")
    show_structure(all_structures[i])

    print(f"Structure {j}, substituted sites drawn originally: {site_sets[j]}")
    show_structure(all_structures[j])
else:
    print("No non-trivial equivalence class was found in this random draw.")

## Final remarks

This notebook illustrates two complementary ideas:

1. **full structural comparison under the symmetry of the pristine crystal**,  
2. **comparison of defect-site patterns under the same symmetry**.

For substitutional defect sampling, the second viewpoint is often the cleanest and most robust.

In a realistic workflow, the recommended strategy is:

- generate many candidate structures,
- remove symmetry-equivalent duplicates,
- optimize the remaining representatives,
- compare their energies and structural motifs.

That way, the final ensemble is both computationally efficient and statistically meaningful.